# model comparison

loads all `results/*_metrics.json` files and builds a side-by-side comparison.
run this any time after adding or updating a model's results file.

models are added to the table as their json files appear — no changes needed here.

In [ ]:
import json
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

AQI_LABELS = ['Good', 'Satisfactory', 'Moderate', 'Poor', 'Very Poor', 'Severe']

In [ ]:
# load all results files
records = []
for path in sorted(glob.glob('results/*_metrics.json')):
    with open(path) as f:
        records.append(json.load(f))

if not records:
    print('no results files found in results/ — run a model notebook first.')
else:
    print(f'loaded {len(records)} model(s): {[r["label"] for r in records]}')

In [ ]:
# build summary table
df = pd.DataFrame(records).set_index('label')

# core metrics first, then per-category MAE columns
core_cols = ['RMSE', 'MAE', 'R2']
cat_cols  = [f'MAE_{c}' for c in AQI_LABELS if f'MAE_{c}' in df.columns]
df = df[[c for c in core_cols + cat_cols if c in df.columns]]

# highlight best value in each column
def highlight_best(col):
    # lower is better for RMSE/MAE; higher is better for R2
    if col.name == 'R2':
        best = col == col.max()
    else:
        best = col == col.min()
    return ['font-weight: bold; color: #1a7a1a' if b else '' for b in best]

df.style \
    .apply(highlight_best) \
    .format({
        'RMSE': '{:.2f}', 'MAE': '{:.2f}', 'R2': '{:.4f}',
        **{c: '{:.1f}' for c in cat_cols}
    }) \
    .set_caption('model comparison — all metrics on original AQI scale')

In [ ]:
# bar chart: RMSE and MAE side by side
models = df.index.tolist()
x      = np.arange(len(models))
width  = 0.35

fig, ax = plt.subplots(figsize=(max(9, len(models) * 1.5), 5))
bars1 = ax.bar(x - width/2, df['RMSE'], width, label='RMSE', color='steelblue',  alpha=0.85)
bars2 = ax.bar(x + width/2, df['MAE'],  width, label='MAE',  color='coral',      alpha=0.85)

for bars in [bars1, bars2]:
    for b in bars:
        ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.5,
                f'{b.get_height():.1f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(models, rotation=20, ha='right')
ax.set(ylabel='AQI units', title='RMSE and MAE by model (lower is better)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# per-category MAE heatmap — shows where each model struggles
cat_df = df[[c for c in cat_cols if c in df.columns]].copy()
cat_df.columns = [c.replace('MAE_', '') for c in cat_df.columns]

if cat_df.shape[1] > 0:
    fig, ax = plt.subplots(figsize=(max(8, cat_df.shape[1] * 1.4), max(3, len(models) * 0.6)))
    im = ax.imshow(cat_df.values.astype(float), aspect='auto', cmap='YlOrRd')
    plt.colorbar(im, ax=ax, label='MAE (AQI units)')
    ax.set_xticks(range(cat_df.shape[1]))
    ax.set_xticklabels(cat_df.columns)
    ax.set_yticks(range(len(models)))
    ax.set_yticklabels(models)
    # annotate cells
    for i, row in enumerate(cat_df.values):
        for j, val in enumerate(row):
            if not np.isnan(val):
                ax.text(j, i, f'{val:.0f}', ha='center', va='center', fontsize=9,
                        color='white' if val > cat_df.values[~np.isnan(cat_df.values)].max() * 0.6 else 'black')
    ax.set(title='MAE by AQI category — darker = larger error')
    plt.tight_layout()
    plt.show()
else:
    print('no per-category data yet — re-run model notebooks to generate it.')

In [ ]:
# r² comparison
fig, ax = plt.subplots(figsize=(max(7, len(models) * 1.3), 4))
colors = ['#2ecc71' if v == df['R2'].max() else 'steelblue' for v in df['R2']]
bars = ax.bar(models, df['R2'], color=colors, edgecolor='black', alpha=0.85)
for b in bars:
    ax.text(b.get_x() + b.get_width() / 2, b.get_height() - 0.01,
            f'{b.get_height():.4f}', ha='center', va='top', fontsize=9, color='white')
ax.set_xticks(range(len(models)))
ax.set_xticklabels(models, rotation=20, ha='right')
ax.set(ylabel='R²', title='R² by model (higher is better)', ylim=(0, 1))
plt.tight_layout()
plt.show()